In [0]:
import pandas as pd
import time
from pyspark.sql import functions as F
from tqdm import tqdm
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
import pyspark
from pyspark.sql import SparkSession
from delta.tables import DeltaTable


In [0]:
# %pip install azure-ai-textanalytics azure-core

In [0]:
endpoint = ""
key = ""

client = TextAnalyticsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

In [0]:
builder = (
    SparkSession.builder
    .appName("sentiment_gold")
)
spark = builder.getOrCreate()
silver_db = "/Volumes/datalake_catalog/datalake_schema/silver"
tickets = (
    spark.read
    .format("delta")
    .load(silver_db+"/support_tickets")
)

In [0]:
tickets.count()

15427

In [0]:
display(tickets)

user_id,ticket_id,category,description,created_at,ingest_time
68823,22257,billing,"Hi Support, Can you send me the invoice for transaction #99283? Create air director though why three hold watch.",2026-03-14T03:24:35.000Z,2026-05-04T07:24:44.220Z
80737,22258,feature_request,"Hi Support, How does the team pricing tier work? Us start start debate effect age top never.",2025-06-08T11:18:54.000Z,2026-05-04T07:24:44.220Z
73227,22259,billing,"Hi Support, Why is my bill higher than the agreed amount? Office indeed agree Congress she road.",2023-05-18T10:46:28.000Z,2026-05-04T07:24:44.220Z
96055,22260,technical,"Hi Support, The dashboard is not loading any data, just a spinning wheel. Pay send total.",2025-01-16T12:39:33.000Z,2026-05-04T07:24:44.220Z
68022,22261,billing,"Hi Support, Can you send me the invoice for transaction #99283? Later true market plant every sure.",2023-10-13T03:50:05.000Z,2026-05-04T07:24:44.220Z
91005,22262,cancellation,"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond. I'm thinking maybe they have a bug. I'm concerned about the security of my {product_purchased} and would like to ensure that my data is safe.",2026-05-06T08:04:04.000Z,2026-05-04T07:24:44.220Z
81361,22264,technical,"Hi Support, The application crashes every time I open the settings tab. Bag finish believe imagine technology challenge.",2024-11-11T16:12:25.000Z,2026-05-04T07:24:44.220Z
66427,22265,feature_request,"Hi Support, Why is there a new device added to my trusted list? Truth day human around he simply product.",2025-08-10T21:22:46.000Z,2026-05-04T07:24:44.220Z
83196,22266,feature_request,"Hi Support, Where is your headquarters located? Themselves attorney southern he agree.",2026-06-07T07:45:49.000Z,2026-05-04T07:24:44.220Z
83853,22267,billing,I'm having an issue with the {product_purchased}. Please assist. (C) 2018-09-19T23:51:03 I'm concerned about the security of my {product_purchased} and would like to ensure that my data is safe.,2024-07-27T04:19:43.000Z,2026-05-04T07:24:44.220Z


In [0]:
tickets.select("ticket_id").distinct().count()

15427

In [0]:
df = tickets.select("ticket_id", "description").limit(1000)

In [0]:
display(df.limit(10))

ticket_id,description
22257,"Hi Support, Can you send me the invoice for transaction #99283? Create air director though why three hold watch."
22258,"Hi Support, How does the team pricing tier work? Us start start debate effect age top never."
22259,"Hi Support, Why is my bill higher than the agreed amount? Office indeed agree Congress she road."
22260,"Hi Support, The dashboard is not loading any data, just a spinning wheel. Pay send total."
22261,"Hi Support, Can you send me the invoice for transaction #99283? Later true market plant every sure."
22262,"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond. I'm thinking maybe they have a bug. I'm concerned about the security of my {product_purchased} and would like to ensure that my data is safe."
22264,"Hi Support, The application crashes every time I open the settings tab. Bag finish believe imagine technology challenge."
22265,"Hi Support, Why is there a new device added to my trusted list? Truth day human around he simply product."
22266,"Hi Support, Where is your headquarters located? Themselves attorney southern he agree."
22267,I'm having an issue with the {product_purchased}. Please assist. (C) 2018-09-19T23:51:03 I'm concerned about the security of my {product_purchased} and would like to ensure that my data is safe.


In [0]:
pdf = (
    df
    .select("ticket_id", "description")
    .dropna(subset=["description"])
    .toPandas()
)

pdf["description"] = pdf["description"].astype(str)
pdf = pdf[pdf["description"].str.strip() != ""].copy()

results = []
batch_size = 10

for i in tqdm(range(0, len(pdf), batch_size)):
    batch = pdf.iloc[i:i + batch_size]

    documents = [
        {
            "id": str(row["ticket_id"]),
            "text": row["description"]
        }
        for _, row in batch.iterrows()
    ]

    response = client.analyze_sentiment(documents=documents)

    for doc in response:
        if doc.is_error:
            results.append({
                "ticket_id": int(doc.id),
                "sentiment": None,
                "positive_score": None,
                "neutral_score": None,
                "negative_score": None,
                "sentiment_error": doc.error.message
            })
        else:
            results.append({
                "ticket_id": int(doc.id),
                "sentiment": doc.sentiment,
                "positive_score": float(doc.confidence_scores.positive),
                "neutral_score": float(doc.confidence_scores.neutral),
                "negative_score": float(doc.confidence_scores.negative),
                "sentiment_error": None
            })

    time.sleep(0.2)

100%|██████████| 100/100 [00:33<00:00,  3.03it/s]


In [0]:
sentiment_pdf = pd.DataFrame(results)
sentiment_spark = spark.createDataFrame(sentiment_pdf)

df_final = df.join(sentiment_spark, on="ticket_id", how="inner").drop("sentiment_error")

display(df_final)

ticket_id,description,sentiment,positive_score,neutral_score,negative_score
22257,"Hi Support, Can you send me the invoice for transaction #99283? Create air director though why three hold watch.",neutral,0.01,0.99,0.01
22258,"Hi Support, How does the team pricing tier work? Us start start debate effect age top never.",neutral,0.05,0.92,0.03
22259,"Hi Support, Why is my bill higher than the agreed amount? Office indeed agree Congress she road.",neutral,0.13,0.83,0.04
22260,"Hi Support, The dashboard is not loading any data, just a spinning wheel. Pay send total.",neutral,0.02,0.97,0.01
22261,"Hi Support, Can you send me the invoice for transaction #99283? Later true market plant every sure.",neutral,0.06,0.94,0.0
22262,"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond. I'm thinking maybe they have a bug. I'm concerned about the security of my {product_purchased} and would like to ensure that my data is safe.",negative,0.0,0.19,0.81
22264,"Hi Support, The application crashes every time I open the settings tab. Bag finish believe imagine technology challenge.",negative,0.0,0.33,0.66
22265,"Hi Support, Why is there a new device added to my trusted list? Truth day human around he simply product.",neutral,0.17,0.82,0.01
22266,"Hi Support, Where is your headquarters located? Themselves attorney southern he agree.",neutral,0.2,0.79,0.0
22267,I'm having an issue with the {product_purchased}. Please assist. (C) 2018-09-19T23:51:03 I'm concerned about the security of my {product_purchased} and would like to ensure that my data is safe.,negative,0.19,0.3,0.51


In [0]:
df_final.count()

1000

In [0]:
def quote_table_name(table_name):
    return ".".join([f"`{part}`" for part in table_name.split(".")])


def get_table_latest_created_at(table_name):
    quoted_table = quote_table_name(table_name)

    details = spark.sql(f"DESCRIBE DETAIL {quoted_table}").collect()[0]
    properties = details["properties"] or {}

    return properties.get("latest_created_at")


def set_table_latest_created_at(table_name, latest_created_at):
    quoted_table = quote_table_name(table_name)

    spark.sql(f"""
        ALTER TABLE {quoted_table}
        SET TBLPROPERTIES (
            'latest_created_at' = '{latest_created_at}'
        )
    """)


def upsert_delta_table(df, table_name, merge_keys):
    """
    Table-level versioned upsert.

    Rules:
    - If table does not exist: save whole dataframe.
    - If table exists:
        - get max(created_at) from current dataframe
        - compare with saved table metadata latest_created_at
        - if current max is not newer: skip
        - if current max is newer: upsert whole dataframe
    """

    if "created_at" not in df.columns:
        raise ValueError("created_at column is required")

    current_latest_created_at = (
        df
        .select(F.max(F.col("created_at")).alias("latest_created_at"))
        .collect()[0]["latest_created_at"]
    )

    if current_latest_created_at is None:
        print(f"Skipped table: {table_name}")
        print("Reason: dataframe has no created_at value")
        print("Inserted rows: 0")
        print("Updated rows: 0")
        return

    current_latest_created_at_str = str(current_latest_created_at)

    if not spark.catalog.tableExists(table_name):
        df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_name)

        set_table_latest_created_at(table_name, current_latest_created_at_str)

        print(f"Created new table: {table_name}")
        print(f"Inserted rows: {df.count()}")
        print("Updated rows: 0")
        print(f"latest_created_at: {current_latest_created_at_str}")
        return

    saved_latest_created_at = get_table_latest_created_at(table_name)

    if saved_latest_created_at is not None:
        should_skip = spark.sql(f"""
            SELECT
                to_timestamp('{current_latest_created_at_str}')
                <= to_timestamp('{saved_latest_created_at}')
                AS should_skip
        """).collect()[0]["should_skip"]

        if should_skip:
            print(f"Skipped table: {table_name}")
            print(f"Reason: current dataframe is not newer than saved table")
            print(f"Current latest created_at: {current_latest_created_at_str}")
            print(f"Saved latest created_at: {saved_latest_created_at}")
            print("Inserted rows: 0")
            print("Updated rows: 0")
            return

    target = DeltaTable.forName(spark, table_name)

    merge_condition = " AND ".join([
        f"target.`{key}` <=> source.`{key}`"
        for key in merge_keys
    ])

    update_set = {
        col: f"source.`{col}`"
        for col in df.columns
    }

    insert_set = {
        col: f"source.`{col}`"
        for col in df.columns
    }

    target.alias("target") \
        .merge(
            df.alias("source"),
            merge_condition
        ) \
        .whenMatchedUpdate(
            condition="to_timestamp(source.`created_at`) > to_timestamp(target.`created_at`)",
            set=update_set
        ) \
        .whenNotMatchedInsert(
            values=insert_set
        ) \
        .execute()

    set_table_latest_created_at(table_name, current_latest_created_at_str)

    history = spark.sql(f"DESCRIBE HISTORY {quote_table_name(table_name)} LIMIT 1")
    metrics = history.select("operationMetrics").collect()[0]["operationMetrics"]

    inserted_rows = int(metrics.get("numTargetRowsInserted", 0))
    updated_rows = int(metrics.get("numTargetRowsUpdated", 0))

    print(f"Upserted table: {table_name}")
    print(f"Inserted rows: {inserted_rows}")
    print(f"Updated rows: {updated_rows}")
    print(f"latest_created_at: {current_latest_created_at_str}")


In [0]:
upsert_delta_table(
    df=df_final.withColumn("created_at", F.current_timestamp()),
    table_name="datalake_catalog.datalake_schema.gold_sentiment_analysis_2",
    merge_keys=["ticket_id"]
)

Created new table: datalake_catalog.datalake_schema.gold_sentiment_analysis_2
Inserted rows: 1000
Updated rows: 0
latest_created_at: 2026-05-04 15:04:02.330885


In [0]:
upsert_delta_table(
    df=df_final.withColumn("created_at", F.current_timestamp()),
    table_name="datalake_catalog.datalake_schema.gold_sentiment_analysis_2",
    merge_keys=["ticket_id"]
)

Upserted table: datalake_catalog.datalake_schema.gold_sentiment_analysis_2
Inserted rows: 0
Updated rows: 0
latest_created_at: 2026-05-04 15:04:16.603941
